# Walkthrough: Bodhan tool server and task agent

A short guided tour. Run `make setup`, fill in `.env` and run `make sample-data` first. Cells 1-3 are free (no Bodhan calls); cell 4 runs one real task (about ₹0.5).

## 1. Connect to the MCP server in-process and list its tools

In [ ]:
import sys, json, asyncio
sys.path.insert(0, "src")
from dotenv import load_dotenv
load_dotenv()

from mcp import Client
from bodhan_mcp.server import build_server

server = build_server()

async def list_tools():
    async with Client(server) as client:
        tools = await client.list_tools()
        for t in tools.tools:
            print(f"{t.name:28} {t.description.splitlines()[0][:90]}")

await list_tools()

## 2. Inspect an input (free)

The agent does this for every input before the planner sees the task.

In [ ]:
async def call(name, **args):
    async with Client(server) as client:
        res = await client.call_tool(name, args)
        return res.structured_content if not res.is_error else res.content[0].text

print(json.dumps(await call("inspect_file", path="sample_data/chapter_scan/chapter.pdf"), indent=2))

## 3. A structured error (free)

Bad input returns a JSON error with a stable `code`, never a crash.

In [ ]:
print(await call("translate_text", target_language="fr", text="Hello"))

## 4. Run the agent on one task

The same command a reviewer runs. Output goes to `outputs/walkthrough/`.

In [ ]:
import subprocess
cmd = [sys.executable, "-m", "bodhan_agent",
       "--task", "Read this paragraph aloud in Bengali with a female voice",
       "--files", "sample_data/paragraph_en.txt",
       "--out", "outputs/walkthrough", "--max-cost-inr", "2"]
subprocess.run(cmd, check=False)

## 5. Read the report and the trace

In [ ]:
from pathlib import Path
out = Path("outputs/walkthrough")
print(Path(out, "result.md").read_text(encoding="utf-8"))
report = json.loads(Path(out, "report.json").read_text(encoding="utf-8"))
print(report.get("status"), report.get("total_cost_inr"))
for line in Path(out, "trace.jsonl").read_text(encoding="utf-8").splitlines():
    step = json.loads(line)
    print(step.get("step"), step.get("type"), step.get("tool"), step.get("latency_s"), step.get("cost_inr"))